# 04 — Analyze Xenium Data

**Spatial Transcriptomics Project | 10x Genomics**

This notebook analyzes data at **single-cell resolution** — a major step beyond Visium spot-level analysis.

**Platform:** 10x Genomics Xenium In-Situ  
**Dataset:** Synthetic dataset mimicking Xenium Human Lung Cancer structure  
**Scale:** 8,000 cells × 100 genes (designed to run on free Colab without downloads)

> **Note on the synthetic data:** The real Xenium Human Lung Cancer dataset from 10x Genomics is ~5GB and requires manual download. This notebook uses a synthetic dataset that exactly replicates the column structure, metadata, and analysis pipeline of the real data. All code is identical to what you would run on the real dataset — only the input data generation differs.

**Key differences from Visium:**

| Feature | Visium | Xenium |
|---|---|---|
| Resolution | ~55 µm spot (multiple cells) | Single cell |
| Detection | Sequencing-based | In-situ hybridization |
| Gene panel | Whole transcriptome | Targeted (~480 genes) |
| Data format | AnnData directly | SpatialData (Zarr) |

**Steps:** Generate data → QC → filter → normalize → cluster → UMAP → spatial scatter → centrality → co-occurrence → neighborhood enrichment → Moran's I

## 1. Install Dependencies

In [ ]:
!pip install squidpy scanpy anndata seaborn igraph leidenalg -q

## 2. Import Packages

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad
import scanpy as sc
import squidpy as sq

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

## 3. Generate Synthetic Xenium-Like Dataset

We simulate a dataset that has exactly the same structure as the real 10x Genomics Xenium Human Lung Cancer output:

- 8,000 cells × 100 genes
- Negative Binomial counts (standard count data model)
- Spatial coordinates over a 10,000 × 10,000 µm tissue field
- All standard Xenium metadata columns: `cell_area`, `nucleus_area`, `control_probe_counts`, etc.
- 5 simulated cell populations with distinct spatial clustering

**Why simulate spatial clustering?** Real Xenium data shows spatially coherent clusters (tumor core, stroma, immune infiltrate). We simulate this by placing cells of the same type in overlapping Gaussian distributions across the tissue.

In [ ]:
np.random.seed(42)

n_cells = 8000
n_genes  = 100
n_cell_types = 5

# --- Assign each cell to one of 5 simulated cell populations ---
cell_type_labels = np.random.choice(n_cell_types, size=n_cells)

# --- Simulate spatial coordinates with population-level clustering ---
# Each population has its own Gaussian center on the tissue
centers = np.array([
    [2000, 2000],
    [8000, 2000],
    [5000, 5000],
    [2000, 8000],
    [8000, 8000],
], dtype=float)

spatial_coords = np.zeros((n_cells, 2))
for ct in range(n_cell_types):
    mask = cell_type_labels == ct
    n = mask.sum()
    spatial_coords[mask] = centers[ct] + np.random.randn(n, 2) * 1500

# Clip to tissue bounds
spatial_coords = np.clip(spatial_coords, 0, 10000)

# --- Simulate gene expression (Negative Binomial) ---
# Each cell type has a distinct expression profile for the first 20 genes
X = np.random.negative_binomial(2, 0.4, size=(n_cells, n_genes)).astype(float)
for ct in range(n_cell_types):
    mask = cell_type_labels == ct
    gene_start = ct * (n_genes // n_cell_types)
    gene_end   = gene_start + (n_genes // n_cell_types)
    # Upregulate marker genes for this cell type
    X[np.ix_(mask, np.arange(gene_start, gene_end))] += np.random.negative_binomial(
        10, 0.3, size=(mask.sum(), gene_end - gene_start)
    )

# --- Build obs metadata matching real Xenium columns ---
cell_ids = [f"cell_{i:06d}" for i in range(n_cells)]

cell_area    = np.random.uniform(50, 300, n_cells)
nucleus_area = cell_area * np.random.uniform(0.2, 0.6, n_cells)
total_counts = X.sum(axis=1)

obs_df = pd.DataFrame({
    "cell_id":                    cell_ids,
    "transcript_counts":          total_counts.astype(int),
    "control_probe_counts":       np.random.binomial(1, 0.0001, n_cells),
    "control_codeword_counts":    np.random.binomial(1, 0.00005, n_cells),
    "unassigned_codeword_counts": np.random.randint(0, 3, n_cells),
    "deprecated_codeword_counts": np.zeros(n_cells, dtype=int),
    "total_counts":               total_counts.astype(int),
    "cell_area":                  cell_area,
    "nucleus_area":               nucleus_area,
    "region":                     ["cell_circles"] * n_cells,
    "z_level":                    np.random.choice([4.0, 5.0, 6.0, 7.0], n_cells),
    "nucleus_count":              np.ones(n_cells, dtype=int),
    "cell_labels":                np.arange(1, n_cells + 1),
}, index=cell_ids)

# --- Gene names matching real Xenium panel style ---
gene_names = (
    ["EPCAM", "KRT7", "KRT19", "KRT8", "MUC1",
     "CDH1", "CLDN4", "CLDN7", "TFF1", "TFF3",
     "AREG", "MET", "ANXA1", "DMBT1", "IGKC",
     "IGHG1", "IDO1", "SPARC", "APOE", "FABP4"]
    + [f"GENE_{i:03d}" for i in range(n_genes - 20)]
)

var_df = pd.DataFrame({
    "gene_ids":     gene_names,
    "feature_types": ["Gene Expression"] * n_genes,
    "genome":        ["GRCh38"] * n_genes,
}, index=gene_names)

# --- Assemble AnnData ---
adata = ad.AnnData(X=X, obs=obs_df, var=var_df)
adata.obsm["spatial"] = spatial_coords

print("Synthetic Xenium-like AnnData:")
print(adata)
print(f"\nCells: {adata.n_obs} | Genes: {adata.n_vars}")
print(f"Spatial coords shape: {adata.obsm['spatial'].shape}")

In [ ]:
# Preview cell metadata (matches real Xenium column structure)
adata.obs.head()

In [ ]:
# Preview spatial coordinates
print("Spatial coordinates (first 5 cells):")
print(adata.obsm["spatial"][:5])

## 4. Quality Control

For Xenium data, key QC metrics are:

| Metric | Meaning | Flag if... |
|---|---|---|
| `total_counts` | Total transcripts per cell | Too few = empty / debris |
| `n_genes_by_counts` | Unique genes detected | Too few = low quality |
| `cell_area` | Cell segmentation area | Too small = debris; too large = doublet |
| Nucleus ratio | nucleus_area / cell_area | Outside 0.2–0.8 range |

We also compute control probe percentage — in real Xenium data this should be < 0.1% (false positive signal rate).

> **Save this plot**

In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50), inplace=True)

# Control probe percentage (should be near 0% in real data)
cprobes = (
    adata.obs["control_probe_counts"].sum() / adata.obs["total_counts"].sum() * 100
)
cwords = (
    adata.obs["control_codeword_counts"].sum() / adata.obs["total_counts"].sum() * 100
)
print(f"Negative DNA probe count %  : {cprobes:.5f}%")
print(f"Negative decoding count %   : {cwords:.5f}%")
print("(Both should be near 0% — confirms high data quality)")

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(16, 4))

axs[0].set_title("Total transcripts per cell")
sns.histplot(adata.obs["total_counts"], kde=False, ax=axs[0])
axs[0].set_xlabel("Total counts")

axs[1].set_title("Unique transcripts per cell")
sns.histplot(adata.obs["n_genes_by_counts"], kde=False, ax=axs[1])
axs[1].set_xlabel("Unique genes detected")

axs[2].set_title("Area of segmented cells")
sns.histplot(adata.obs["cell_area"], kde=False, ax=axs[2])
axs[2].set_xlabel("Cell area (µm²)")

axs[3].set_title("Nucleus / Cell area ratio")
sns.histplot(
    adata.obs["nucleus_area"] / adata.obs["cell_area"],
    kde=False, ax=axs[3]
)
axs[3].set_xlabel("Nucleus ratio")

plt.tight_layout()
plt.savefig("xenium_qc_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: xenium_qc_distributions.png")

## 5. Filter, Normalize & Cluster

Same pipeline as notebooks 01–03 — demonstrating that the Scanpy toolkit is fully platform-agnostic:

1. **Filter cells** with fewer than 10 counts
2. **Filter genes** present in fewer than 5 cells
3. **Normalize** counts per cell
4. **Log-transform**
5. **PCA** → **KNN graph** → **UMAP** → **Leiden clustering**

In [ ]:
print(f"Before filtering: {adata.n_obs} cells, {adata.n_vars} genes")

sc.pp.filter_cells(adata, min_counts=10)
sc.pp.filter_genes(adata, min_cells=5)

print(f"After filtering:  {adata.n_obs} cells, {adata.n_vars} genes")

In [ ]:
# Save raw counts before normalization
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(adata, flavor="igraph", n_iterations=2, directed=False)

print(f"Leiden clusters found: {adata.obs['leiden'].nunique()}")
print("Cluster sizes:")
print(adata.obs["leiden"].value_counts().sort_index())

## 6. UMAP Visualization

UMAP projects high-dimensional gene expression into 2D. Cells with similar expression profiles cluster together. Unlike spatial scatter, UMAP removes physical location information — it shows **transcriptional** similarity.

> **Save this plot**

In [ ]:
sc.pl.umap(
    adata,
    color=["total_counts", "n_genes_by_counts", "leiden"],
    wspace=0.4,
    show=False,
)
plt.savefig("xenium_umap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: xenium_umap.png")

## 7. Spatial Scatter — Leiden Clusters on Tissue

Unlike UMAP, spatial scatter overlays clusters on physical tissue coordinates. This shows **where** each cell type lives in the tissue.

**Key difference from Visium:** Here each dot = one cell (not a ~55µm spot containing multiple cells).

> **Save this plot**

In [ ]:
sq.pl.spatial_scatter(
    adata,
    shape=None,
    color=["leiden"],
    wspace=0.4,
    size=5,
)
plt.savefig("xenium_spatial_leiden.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: xenium_spatial_leiden.png")

## 8. Build Spatial Neighborhood Graph (Delaunay)

For single-cell data, we use **Delaunay triangulation** to build the spatial graph (instead of a fixed grid as in Visium). Delaunay connects cells that are geometrically adjacent without pre-specifying a radius.

Three centrality scores are then computed per cluster:
- **Closeness centrality**: how central/accessible is this cluster?
- **Clustering coefficient**: how densely do cells of this type group together?
- **Degree centrality**: fraction of other cells each cluster is connected to

> **Save this plot**

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type="generic", delaunay=True)
print(f"Spatial graph built. Connectivity matrix: {adata.obsp['spatial_connectivities'].shape}")

In [ ]:
sq.gr.centrality_scores(adata, cluster_key="leiden")

sq.pl.centrality_scores(adata, cluster_key="leiden", figsize=(16, 5))
plt.savefig("xenium_centrality_scores.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: xenium_centrality_scores.png")

## 9. Co-occurrence Probability

At single-cell resolution, co-occurrence gives fine-grained information about which clusters physically neighbor each other across increasing spatial distances.

$$\text{Score} = \frac{p(\text{exp} \mid \text{cond})}{p(\text{exp})}$$

We work on a 50% subsample to speed up computation.

> **Save these plots**

In [ ]:
# Create 50% subsample for faster co-occurrence computation
adata_subsample = sc.pp.subsample(adata, fraction=0.5, copy=True)
print(f"Subsample size: {adata_subsample.n_obs} cells")

In [ ]:
sq.gr.co_occurrence(adata_subsample, cluster_key="leiden")

# Pick the first cluster to visualize
first_cluster = adata_subsample.obs["leiden"].cat.categories[0]
print(f"Showing co-occurrence for cluster: {first_cluster}")

sq.pl.co_occurrence(
    adata_subsample,
    cluster_key="leiden",
    clusters=first_cluster,
    figsize=(10, 6),
)
plt.savefig("xenium_co_occurrence.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: xenium_co_occurrence.png")

In [ ]:
sq.pl.spatial_scatter(
    adata_subsample,
    color="leiden",
    shape=None,
    size=5,
)
plt.savefig("xenium_subsample_spatial.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: xenium_subsample_spatial.png")

## 10. Neighborhood Enrichment

Which Leiden clusters are spatial neighbors more often than expected by chance? Score > 0 = enriched proximity, Score < 0 = spatial avoidance.

> **Save this plot**

In [ ]:
sq.gr.nhood_enrichment(adata, cluster_key="leiden")

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

sq.pl.nhood_enrichment(
    adata,
    cluster_key="leiden",
    title="Neighborhood enrichment",
    ax=ax[0],
)

sq.pl.spatial_scatter(
    adata_subsample,
    color="leiden",
    shape=None,
    size=5,
    ax=ax[1],
)

plt.tight_layout()
plt.savefig("xenium_nhood_enrichment.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: xenium_nhood_enrichment.png")

## 11. Spatially Variable Genes — Moran's I

At single-cell resolution, Moran's I identifies genes whose expression is spatially patterned. In real Xenium lung cancer data, top genes like AREG, MET, ANXA1, and EPCAM show strong spatial patterns (tumor islands, immune infiltrate, stroma).

We rebuild the spatial graph on the subsample for this analysis.

> **Save this plot**

In [ ]:
# Rebuild spatial graph on subsample
sq.gr.spatial_neighbors(adata_subsample, coord_type="generic", delaunay=True)

# Compute Moran's I for all genes
sq.gr.spatial_autocorr(
    adata_subsample,
    mode="moran",
    n_perms=100,
    n_jobs=1,
)

print("Top 10 spatially variable genes (Moran's I):")
print(adata_subsample.uns["moranI"].head(10))

In [ ]:
top_genes = adata_subsample.uns["moranI"].head(2).index.tolist()
print(f"Top spatially variable genes: {top_genes}")

sq.pl.spatial_scatter(
    adata_subsample,
    color=top_genes,
    shape=None,
    size=5,
    img=False,
)
plt.savefig("xenium_spatially_variable_genes.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: xenium_spatially_variable_genes.png")

## 12. Summary & Key Findings

| Analysis | Key Finding |
|---|---|
| QC | Very low control probe rates → high data quality |
| Leiden clustering | Distinct transcriptional populations identified |
| UMAP | Clear separation of simulated cell populations |
| Spatial scatter | Clusters show spatial coherence (same type in same region) |
| Centrality | Different clusters have distinct spatial network properties |
| Co-occurrence | Fine-grained cell-cell proximity patterns at single-cell scale |
| Moran's I | Top marker genes show strong spatial patterning |

**How Xenium connects back to Visium:**

| Aspect | Connection |
|---|---|
| Scanpy pipeline | Identical QC → normalize → cluster workflow |
| Squidpy functions | Same `co_occurrence`, `nhood_enrichment`, `spatial_autocorr` |
| Resolution upgrade | Visium spots (~55µm, multiple cells) → Xenium (single cell) |
| Deconvolution | Xenium cell-type data can deconvolve Visium spot mixtures |
| Data format | Visium: AnnData directly. Xenium: SpatialData (Zarr) wrapping AnnData |

---
**Project complete!** All 4 notebooks form a full spatial transcriptomics workflow.

In [ ]:
# Save processed AnnData
adata.write_h5ad("xenium_synthetic_processed.h5ad")
print("Saved: xenium_synthetic_processed.h5ad")
print("\nAll figures to download for GitHub:")
for f in [
    "xenium_qc_distributions.png",
    "xenium_umap.png",
    "xenium_spatial_leiden.png",
    "xenium_centrality_scores.png",
    "xenium_co_occurrence.png",
    "xenium_subsample_spatial.png",
    "xenium_nhood_enrichment.png",
    "xenium_spatially_variable_genes.png",
]:
    print(f"  - {f}")